# Qwen2.5-VL-7B original-image split pipeline

1. Runs model inference for **all rows**: `ID` and `Counting/Illusion`.
2. Runs token-level log-prob signal extraction **only for numeric rows** where the ground-truth and comparison candidates are **single-token** under the Qwen tokenizer.

Outputs:

- `original_model_outputs.csv` — all rows, model outputs only.
- `original_numeric_signal.csv` — numeric rows with valid single-token GT and single-token competitors.
- `original_numeric_candidates.csv` — one row per valid single-token candidate.
- `original_numeric_signal_skipped.csv` — numeric rows skipped because GT/candidates were multi-token or missing.


In [1]:
# Check GPU.
!nvidia-smi

Thu May 14 14:47:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install dependencies. Restart runtime only if Colab asks you to.
!pip install -q transformers accelerate pillow datasets

In [3]:
import gc
import re
import json
import math
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm
from PIL import Image

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

### Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# HuggingFace login
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Configuration


In [35]:
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

DATASET_NAME = "anvo25/vlms-are-biased"
SPLIT_NAME = "original"

OUTPUT_DIR = Path("/content/gdrive/MyDrive/original_token_outputs/qween7b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Original split has no expected_bias, so proxy bias comes from GT±WINDOW.
WINDOW = 2
MIN_CANDIDATE = 0

EXCLUDE_OPTICAL_ILLUSIONS = True

# Use 20 for testing. Set None for the full original split.
MAX_SAMPLES = None

# Batching controls.
BATCH_SIZE_NUMERIC = 1
BATCH_SIZE_ID = 1

# Generation length.
# Numeric rows are short; ID rows need more tokens.
MAX_NEW_TOKENS_NUMERIC = 3
MAX_NEW_TOKENS_ID = 32

# Numeric suffix for counting rows.
# Use "integer", not "digit", because some GT values may be multi-digit
# but still single-token under the tokenizer.
NUMERIC_SUFFIX = "\nAnswer with exactly ONE digit (0-9). No text, no multiple digits."

# Save intermediate CSVs every N processed rows.
SAVE_EVERY = 20

print("MODEL_ID:", MODEL_ID)
print("DATASET:", DATASET_NAME, SPLIT_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("WINDOW:", WINDOW)
print("BATCH_SIZE_NUMERIC:", BATCH_SIZE_NUMERIC)
print("BATCH_SIZE_ID:", BATCH_SIZE_ID)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

MODEL_ID: Qwen/Qwen2.5-VL-7B-Instruct
DATASET: anvo25/vlms-are-biased original
OUTPUT_DIR: /content/gdrive/MyDrive/original_token_outputs/qween7b
WINDOW: 2
BATCH_SIZE_NUMERIC: 1
BATCH_SIZE_ID: 1
torch: 2.10.0+cu128
cuda available: True


## Load Model

In [7]:
# Load model and processor
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [8]:
# Clean memory before loading the 7B model.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if device == "cuda" else None,
    attn_implementation="sdpa",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Required for batched decoder-only generation.
processor.tokenizer.padding_side = "left"
tokenizer = processor.tokenizer

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False  # lower memory use

model.eval()

print("Loaded model:", MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded model: Qwen/Qwen2.5-VL-7B-Instruct


## Load original split from HuggingFace

In [36]:
def load_original_dataset():
    """
    Load original split, apply filtering/subsetting, then reset idx.
    """
    ds = load_dataset(DATASET_NAME, split=SPLIT_NAME)

    print("Original dataset length:", len(ds))
    print("Columns before filtering:", ds.column_names)

    if EXCLUDE_OPTICAL_ILLUSIONS:
        before = len(ds)
        ds = ds.filter(
            lambda x: str(x.get("topic", "")).strip().lower() != "optical illusion"
        )
        print(f"Removed Optical Illusion rows: {before - len(ds)}")

    if MAX_SAMPLES is not None:
        ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
        print("Using debug subset:", len(ds))

    # Critical index reset.
    if "idx" in ds.column_names:
        ds = ds.remove_columns(["idx"])

    ds = ds.add_column("idx", list(range(len(ds))))

    # Validate idx immediately.
    idx_values = ds["idx"]
    assert len(idx_values) == len(ds)
    assert idx_values[0] == 0
    assert idx_values[-1] == len(ds) - 1
    assert len(set(idx_values)) == len(ds)

    print("Final dataset length:", len(ds))
    print("Columns after indexing:", ds.column_names)

    print("\nTopic counts:")
    for topic, count in Counter(ds["topic"]).items():
        print(f"{topic}: {count}")

    return ds

In [37]:
dataset = load_original_dataset()

Original dataset length: 458
Columns before filtering: ['image', 'ID', 'image_path', 'topic', 'sub_topic', 'prompt', 'ground_truth', 'expected_bias', 'with_title', 'type_of_question', 'pixel', 'metadata']
Removed Optical Illusion rows: 132


Flattening the indices:   0%|          | 0/326 [00:00<?, ? examples/s]

Final dataset length: 326
Columns after indexing: ['image', 'ID', 'image_path', 'topic', 'sub_topic', 'prompt', 'ground_truth', 'expected_bias', 'with_title', 'type_of_question', 'pixel', 'metadata', 'idx']

Topic counts:
Flags: 38
Animals: 182
Logos: 94
Game Boards: 8
Chess Pieces: 4


## Helper Functions

In [11]:
_BRACE_RE = re.compile(r"\{([^{}]+)\}")
_INT_RE = re.compile(r"-?\d+")


def clean_text(x):
    """Convert None/NaN to empty string and strip whitespace."""
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()

In [12]:
def unbrace(x):
    """Return content inside {...} if present, otherwise stripped text."""
    x = clean_text(x)
    m = _BRACE_RE.search(x)
    return m.group(1).strip() if m else x

In [13]:
def first_int(x):
    """Extract first integer from text, or None if no integer exists."""
    m = _INT_RE.search(unbrace(x))
    return int(m.group(0)) if m else None

In [14]:
def norm_answer(x):
    """Simple normalization for exact-match diagnostics."""
    return unbrace(x).lower().strip().rstrip(".")

In [15]:
def is_numeric_sample(sample):
    """
    True for rows eligible for token-level numeric signal extraction.
    ID rows are generated but not signal-analyzed.
    """
    qtype = clean_text(sample.get("type_of_question", "")).lower()
    return ("count" in qtype or "illusion" in qtype) and first_int(sample.get("ground_truth")) is not None

In [16]:
def candidate_window(gt):
    """Build GT±WINDOW candidate list."""
    gt = int(gt)
    return list(range(max(MIN_CANDIDATE, gt - WINDOW), gt + WINDOW + 1))

In [17]:
def single_token_meta(value):
    """
    Return token metadata if value has a one-token representation.
    Otherwise return None.

    We test variants because Qwen may tokenize answer-formatted numbers differently.
    """
    s = str(value)

    for variant in (s, " " + s, "{" + s, "{" + s + "}"):
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if len(ids) == 1:
            return {
                "candidate": int(value),
                "token_id": int(ids[0]),
                "token_variant": variant,
            }

    return None

In [18]:
def build_single_token_window(gt):
    """Build GT±WINDOW and split candidates into valid single-token vs skipped."""
    full = candidate_window(gt)
    valid = []
    skipped = []

    for c in full:
        meta = single_token_meta(c)
        if meta is None:
            skipped.append(int(c))
        else:
            valid.append(meta)

    return full, valid, skipped

In [19]:
def decode_one_token(token_id):
    """Decode a single token id."""
    return tokenizer.decode([int(token_id)], skip_special_tokens=True)


def find_answer_step(gen_token_ids):
    """
    Find first generated token containing an integer.

    Returns:
        answer_step, detected_digit, detected_token_text
    """
    if gen_token_ids is None:
        return None, None, None

    for step, token_id in enumerate(gen_token_ids):
        token_text = decode_one_token(token_id)
        value = first_int(token_text)
        if value is not None:
            return step, value, token_text

    return None, None, None

In [20]:
def qwen_text(image, prompt):
    """Apply Qwen chat template to one image/prompt pair."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


In [21]:
def make_base_row(sample, prompt_used, answer, error=""):
    """Shared output fields for all CSVs."""
    gt_raw = sample.get("ground_truth", "")
    pred_num = first_int(answer)
    gt_num = first_int(gt_raw)

    return {
        "idx": int(sample["idx"]),
        "ID": sample.get("ID"),
        "image_path": sample.get("image_path"),
        "topic": sample.get("topic"),
        "sub_topic": sample.get("sub_topic"),
        "type_of_question": clean_text(sample.get("type_of_question", "")),
        "prompt": clean_text(sample.get("prompt", "")),
        "prompt_used_for_generation": prompt_used,
        "ground_truth": gt_raw,
        "ground_truth_num": gt_num,
        "model_answer": answer,
        "pred_norm": norm_answer(answer),
        "gt_norm": norm_answer(gt_raw),
        "pred_num": pred_num,
        "correct_exact": norm_answer(answer) == norm_answer(gt_raw),
        "correct_numeric": (
            int(pred_num) == int(gt_num)
            if pred_num is not None and gt_num is not None
            else False
        ),
        "generation_error": error,
    }


def sort_key(row):
    """Safe sort by idx, with missing idx pushed to the end."""
    idx = row.get("idx")
    return 10**12 if idx is None else int(idx)

### Save Outputs functions

In [38]:
all_output_rows = []
token_signal_rows = []
token_candidate_rows = []
skipped_rows = []

In [39]:
def save_outputs():
    """Save current results to CSV."""
    pd.DataFrame(sorted(all_output_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_model_outputs.csv",
        index=False,
    )

    pd.DataFrame(sorted(token_signal_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_numeric_signal.csv",
        index=False,
    )

    pd.DataFrame(
        sorted(
            token_candidate_rows,
            key=lambda r: (sort_key(r), int(r.get("candidate", -1))),
        )
    ).to_csv(
        OUTPUT_DIR / "original_numeric_candidates.csv",
        index=False,
    )

    pd.DataFrame(sorted(skipped_rows, key=sort_key)).to_csv(
        OUTPUT_DIR / "original_numeric_signal_skipped.csv",
        index=False,
    )

## Benchmark Generation

In [40]:
def generate_batch(samples, max_new_tokens, need_scores):
    """
    7B-safe generation. Batch size is configured as 1 by default.
    """
    prepared = []

    for sample in samples:
        sample = dict(sample)
        prompt_raw = clean_text(sample.get("prompt", ""))

        prompt_used = (
            prompt_raw + NUMERIC_SUFFIX
            if is_numeric_sample(sample)
            else prompt_raw
        )

        try:
            image = sample["image"].convert("RGB")
            text = qwen_text(image, prompt_used)

            prepared.append({
                "sample": sample,
                "prompt_used": prompt_used,
                "image": image,
                "text": text,
                "error": "",
            })

        except Exception as e:
            prepared.append({
                "sample": sample,
                "prompt_used": prompt_used,
                "image": None,
                "text": None,
                "error": repr(e),
            })

    results = [None] * len(prepared)
    good_indices = [i for i, x in enumerate(prepared) if not x["error"]]

    for i, item in enumerate(prepared):
        if item["error"]:
            results[i] = {
                "sample": item["sample"],
                "prompt_used": item["prompt_used"],
                "answer": "",
                "gen_token_ids": None,
                "log_probs": None,
                "error": item["error"],
            }

    if not good_indices:
        return results

    texts = [prepared[i]["text"] for i in good_indices]
    images = [prepared[i]["image"] for i in good_indices]

    inputs = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=need_scores,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False,
        )

    prompt_len = inputs["input_ids"].shape[1]

    batch_log_probs = None
    if need_scores and len(output.scores) > 0:
        scores = torch.stack(output.scores, dim=0).float()
        batch_log_probs = torch.log_softmax(scores, dim=-1)

    for local_b, original_i in enumerate(good_indices):
        gen_ids = output.sequences[local_b, prompt_len:].detach().cpu()

        answer = tokenizer.decode(
            gen_ids,
            skip_special_tokens=True,
        ).strip()

        if need_scores:
            ex_log_probs = batch_log_probs[:, local_b, :].detach().cpu()
        else:
            ex_log_probs = None

        results[original_i] = {
            "sample": prepared[original_i]["sample"],
            "prompt_used": prepared[original_i]["prompt_used"],
            "answer": answer,
            "gen_token_ids": gen_ids,
            "log_probs": ex_log_probs,
            "error": "",
        }

    del inputs, output
    if batch_log_probs is not None:
        del batch_log_probs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

### Token-level analysis for one numeric row

In [41]:
def analyze_numeric_result(result):
    """
    Compute token-level signal for one numeric row.

    Signal:
        margin_token = logP(GT) - logP(best non-GT candidate in GT±WINDOW)
    """
    sample = result["sample"]
    gt_num = first_int(sample.get("ground_truth", ""))

    if gt_num is None:
        return None, [], {"skip_reason": "non_numeric_ground_truth"}

    if single_token_meta(gt_num) is None:
        return None, [], {
            "skip_reason": "gt_not_single_token",
            "ground_truth_num": gt_num,
        }

    full_candidates, valid_candidates, skipped_candidates = build_single_token_window(gt_num)

    competitors = [
        c for c in valid_candidates
        if int(c["candidate"]) != int(gt_num)
    ]

    if not competitors:
        return None, [], {
            "skip_reason": "no_single_token_competitor_in_window",
            "ground_truth_num": gt_num,
            "candidate_set": json.dumps(full_candidates),
            "skipped_multitoken_candidates": json.dumps(skipped_candidates),
        }

    answer_step, detected_digit, detected_token_text = find_answer_step(
        result["gen_token_ids"]
    )

    if answer_step is None:
        return None, [], {
            "skip_reason": "no_numeric_answer_step",
            "ground_truth_num": gt_num,
            "candidate_set": json.dumps(full_candidates),
            "skipped_multitoken_candidates": json.dumps(skipped_candidates),
        }

    log_probs = result["log_probs"]

    if log_probs is None or answer_step >= log_probs.shape[0]:
        return None, [], {
            "skip_reason": "missing_generation_scores",
            "ground_truth_num": gt_num,
            "candidate_set": json.dumps(full_candidates),
            "skipped_multitoken_candidates": json.dumps(skipped_candidates),
        }

    # Full-vocab log-probs at exact answer step.
    lp = log_probs[answer_step].float()

    # Full-vocab entropy.
    probs = lp.exp()
    entropy_token = float(-(probs * lp).sum().item())

    candidate_rows = []
    for meta in valid_candidates:
        candidate_rows.append({
            **meta,
            "logprob_token": float(lp[meta["token_id"]].item()),
            "is_gt": int(meta["candidate"]) == int(gt_num),
        })

    gt_rows = [
        r for r in candidate_rows
        if int(r["candidate"]) == int(gt_num)
    ]

    if not gt_rows:
        return None, candidate_rows, {
            "skip_reason": "gt_missing_from_valid_candidates",
            "ground_truth_num": gt_num,
            "candidate_set": json.dumps(full_candidates),
            "skipped_multitoken_candidates": json.dumps(skipped_candidates),
        }

    gt_row = gt_rows[0]

    proxy_bias = max(
        [r for r in candidate_rows if int(r["candidate"]) != int(gt_num)],
        key=lambda r: r["logprob_token"],
    )

    sorted_window = sorted(
        candidate_rows,
        key=lambda r: r["logprob_token"],
        reverse=True,
    )

    gt_rank_window_token = next(
        i + 1
        for i, r in enumerate(sorted_window)
        if int(r["candidate"]) == int(gt_num)
    )

    margin_token = float(
        gt_row["logprob_token"] - proxy_bias["logprob_token"]
    )

    base = make_base_row(
        sample=sample,
        prompt_used=result["prompt_used"],
        answer=result["answer"],
        error=result["error"],
    )

    signal_row = {
        **base,

        "detected_digit": detected_digit,
        "detected_token_text": detected_token_text,
        "answer_step": answer_step,

        "candidate_policy": f"GT±{WINDOW}, single-token only",
        "candidate_set": json.dumps(full_candidates),
        "valid_single_token_candidates": json.dumps(
            [int(c["candidate"]) for c in valid_candidates]
        ),
        "skipped_multitoken_candidates": json.dumps(skipped_candidates),

        "gt_token_id": gt_row["token_id"],
        "gt_token_variant": gt_row["token_variant"],
        "gt_logprob_token": gt_row["logprob_token"],

        "proxy_bias_value": proxy_bias["candidate"],
        "proxy_bias_token_id": proxy_bias["token_id"],
        "proxy_bias_token_variant": proxy_bias["token_variant"],
        "proxy_bias_logprob_token": proxy_bias["logprob_token"],

        "margin_token": margin_token,
        "prefers_gt_token": margin_token > 0,
        "gt_rank_window_token": gt_rank_window_token,
        "entropy_token": entropy_token,

        "num_valid_single_token_candidates": len(valid_candidates),
        "num_skipped_multitoken_candidates": len(skipped_candidates),
    }

    return signal_row, candidate_rows, None

## Split rows and run batches

In [42]:
numeric_samples = []
id_samples = []

for sample in dataset:
    sample = dict(sample)

    # Defensive index validation.
    if sample.get("idx") is None:
        raise ValueError(f"Missing idx in sample: {sample}")

    if is_numeric_sample(sample):
        numeric_samples.append(sample)
    else:
        id_samples.append(sample)

print("Numeric rows:", len(numeric_samples))
print("ID/text rows:", len(id_samples))

Numeric rows: 163
ID/text rows: 163


In [43]:
def iter_batches(rows, batch_size):
    for start in range(0, len(rows), batch_size):
        yield rows[start:start + batch_size]


processed = 0
next_save_at = SAVE_EVERY

## Numerical Rows

In [44]:
for batch in tqdm(
    iter_batches(numeric_samples, BATCH_SIZE_NUMERIC),
    total=int(np.ceil(len(numeric_samples) / BATCH_SIZE_NUMERIC)),
    desc="Numeric batches",
):
    batch_results = generate_batch(
        samples=batch,
        max_new_tokens=MAX_NEW_TOKENS_NUMERIC,
        need_scores=True,
    )

    for result in batch_results:
        sample = result["sample"]

        base = make_base_row(
            sample=sample,
            prompt_used=result["prompt_used"],
            answer=result["answer"],
            error=result["error"],
        )

        all_output_rows.append(base)

        if result["error"]:
            skipped_rows.append({
                **base,
                "pipeline": "token",
                "skip_reason": "generation_error",
            })
            processed += 1
            continue

        signal_row, candidate_rows, skip_info = analyze_numeric_result(result)

        for c in candidate_rows:
            token_candidate_rows.append({
                "idx": int(sample["idx"]),
                "ID": sample.get("ID"),
                "topic": sample.get("topic"),
                "sub_topic": sample.get("sub_topic"),
                "ground_truth_num": first_int(sample.get("ground_truth")),
                **c,
            })

        if signal_row is not None:
            token_signal_rows.append(signal_row)
        else:
            skipped_rows.append({
                **base,
                "pipeline": "token",
                **(skip_info or {}),
            })

        processed += 1

    if processed >= next_save_at:
        save_outputs()
        print(f"Intermediate save at {processed} processed rows.")
        next_save_at += SAVE_EVERY

    gc.collect()

Numeric batches:   0%|          | 0/163 [00:00<?, ?it/s]

Intermediate save at 20 processed rows.
Intermediate save at 40 processed rows.
Intermediate save at 60 processed rows.
Intermediate save at 80 processed rows.
Intermediate save at 100 processed rows.
Intermediate save at 120 processed rows.
Intermediate save at 140 processed rows.
Intermediate save at 160 processed rows.


## ID rows

In [45]:
for batch in tqdm(
    iter_batches(id_samples, BATCH_SIZE_ID),
    total=int(np.ceil(len(id_samples) / BATCH_SIZE_ID)),
    desc="ID/text batches",
):
    batch_results = generate_batch(
        samples=batch,
        max_new_tokens=MAX_NEW_TOKENS_ID,
        need_scores=False,
    )

    for result in batch_results:
        sample = result["sample"]

        base = make_base_row(
            sample=sample,
            prompt_used=result["prompt_used"],
            answer=result["answer"],
            error=result["error"],
        )

        all_output_rows.append(base)

        if result["error"]:
            skipped_rows.append({
                **base,
                "pipeline": "generation",
                "skip_reason": "generation_error",
            })

        processed += 1

    if processed >= next_save_at:
        save_outputs()
        print(f"Intermediate save at {processed} processed rows.")
        next_save_at += SAVE_EVERY

    gc.collect()

ID/text batches:   0%|          | 0/163 [00:00<?, ?it/s]

Intermediate save at 180 processed rows.
Intermediate save at 200 processed rows.
Intermediate save at 220 processed rows.
Intermediate save at 240 processed rows.
Intermediate save at 260 processed rows.
Intermediate save at 280 processed rows.
Intermediate save at 300 processed rows.
Intermediate save at 320 processed rows.


## Save and validate

In [46]:
# Final defensive sort.
all_output_rows = sorted(all_output_rows, key=sort_key)
token_signal_rows = sorted(token_signal_rows, key=sort_key)
token_candidate_rows = sorted(
    token_candidate_rows,
    key=lambda r: (sort_key(r), int(r.get("candidate", -1))),
)
skipped_rows = sorted(skipped_rows, key=sort_key)

In [47]:
# Validate index health.
print("Missing idx counts:")
print("all_output_rows:", sum(r.get("idx") is None for r in all_output_rows))
print("token_signal_rows:", sum(r.get("idx") is None for r in token_signal_rows))
print("token_candidate_rows:", sum(r.get("idx") is None for r in token_candidate_rows))
print("skipped_rows:", sum(r.get("idx") is None for r in skipped_rows))

Missing idx counts:
all_output_rows: 0
token_signal_rows: 0
token_candidate_rows: 0
skipped_rows: 0


In [48]:
# Validate row coverage.
expected_rows = len(dataset)
actual_rows = len(all_output_rows)
print("Expected model output rows:", expected_rows)
print("Actual model output rows:", actual_rows)

assert actual_rows == expected_rows, (
    f"Expected {expected_rows} all-output rows, got {actual_rows}"
)

Expected model output rows: 326
Actual model output rows: 326


In [49]:
save_outputs()

# Save parquet copies too.
pd.DataFrame(all_output_rows).to_parquet(
    OUTPUT_DIR / "original_model_outputs.parquet",
    index=False,
)
pd.DataFrame(token_signal_rows).to_parquet(
    OUTPUT_DIR / "original_numeric_signal.parquet",
    index=False,
)
pd.DataFrame(token_candidate_rows).to_parquet(
    OUTPUT_DIR / "original_numeric_candidates.parquet",
    index=False,
)
pd.DataFrame(skipped_rows).to_parquet(
    OUTPUT_DIR / "original_numeric_signal_skipped.parquet",
    index=False,
)

In [50]:
print("\nDone.")
print("All output rows:", len(all_output_rows))
print("Token signal rows:", len(token_signal_rows))
print("Token candidate rows:", len(token_candidate_rows))
print("Skipped rows:", len(skipped_rows))

print("\nSaved files:")
print(OUTPUT_DIR / "original_model_outputs.csv")
print(OUTPUT_DIR / "original_numeric_signal.csv")
print(OUTPUT_DIR / "original_numeric_candidates.csv")
print(OUTPUT_DIR / "original_numeric_signal_skipped.csv")


Done.
All output rows: 326
Token signal rows: 154
Token candidate rows: 761
Skipped rows: 9

Saved files:
/content/gdrive/MyDrive/original_token_outputs/qween7b/original_model_outputs.csv
/content/gdrive/MyDrive/original_token_outputs/qween7b/original_numeric_signal.csv
/content/gdrive/MyDrive/original_token_outputs/qween7b/original_numeric_candidates.csv
/content/gdrive/MyDrive/original_token_outputs/qween7b/original_numeric_signal_skipped.csv
